In [ ]:
REPO_URL = "https://github.com/huyvanzzz/finetune-InternVL2.git"
BRANCH = "feature/sailvl-core"
PROJECT_DIR = "/kaggle/working/finetune-InternVL2"
CONFIG_PATH = "sailvl_config.yaml"


In [ ]:
import os, subprocess, pathlib
if not pathlib.Path(PROJECT_DIR).exists():
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
os.chdir(PROJECT_DIR)
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "reset", "--hard", f"origin/{BRANCH}"], check=True)
print("Current repo:", os.getcwd())


In [ ]:
!pip uninstall -y torch torchvision torchaudio triton flash-attn torchao peft || true
!pip install -q --no-cache-dir --index-url https://download.pytorch.org/whl/cu126 torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1
!pip install -q --no-cache-dir bitsandbytes transformers==4.46.2 datasets accelerate timm evaluate rouge_score scikit-learn safetensors huggingface_hub sentencepiece mlflow pyyaml pillow tqdm peft==0.18.1


In [ ]:
import torch, transformers, peft
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
!nvidia-smi
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    print('capability:', torch.cuda.get_device_capability(0))
    x = torch.zeros(1, device='cuda', dtype=torch.float16)
    print('cuda fp16 tensor ok:', x.dtype)
!python -m py_compile train.py wad_dataset.py scripts/test_infer.py scripts/prepare_qformer.py scripts/smoke_sail_backend.py model_backends/sailvl/runtime.py model_backends/sailvl/preprocess.py model_backends/sailvl/qformer_bridge.py


In [ ]:
import yaml
with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)
OUTPUT_DIR = cfg["training"]["output_dir"]
print('model:', cfg['model']['name'])
print('architecture:', cfg['model']['architecture'])
print('qformer enabled:', cfg['model']['qformer']['enabled'])
print('output_dir:', OUTPUT_DIR)
print('max_dynamic_patch:', cfg['model']['vision']['max_dynamic_patch'])
print('prompt_mode:', cfg['data']['direct_text_alter_prompt_mode'])


In [ ]:
import subprocess
subprocess.run(["python", "build_frame_index.py"], input="n\n", text=True, check=True)


In [ ]:
import subprocess
subprocess.run(["python", "scripts/prepare_qformer.py", "--config", CONFIG_PATH], check=True)


In [ ]:
import subprocess
SMOKE_CHECKPOINT = ""  # optional: local folder hoac HF repo id
cmd = ["python", "scripts/smoke_sail_backend.py", "--config", CONFIG_PATH]
if SMOKE_CHECKPOINT:
    cmd += ["--checkpoint", SMOKE_CHECKPOINT]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
import subprocess

TRAIN_CHECKPOINT = ""  # local folder hoac HF repo id; de rong neu train tu dau
TRAIN_START_EPOCH = None
TRAIN_START_STEP = None

cmd = ["python", "train.py", "--config", CONFIG_PATH]
if TRAIN_CHECKPOINT:
    cmd += ["--checkpoint", TRAIN_CHECKPOINT]
if TRAIN_START_EPOCH is not None:
    cmd += ["--start_epoch", str(TRAIN_START_EPOCH)]
if TRAIN_START_STEP is not None:
    cmd += ["--start_step", str(TRAIN_START_STEP)]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
import subprocess
subprocess.run([
    "python", "scripts/fit_tfidf.py",
    "--config", CONFIG_PATH,
    "--output", "tfidf_vectorizer.pkl",
], check=True)


In [ ]:
import subprocess
from pathlib import Path

CHECKPOINT_DIR = ""  # optional: fill local path hoac HF repo id
if not CHECKPOINT_DIR:
    candidates = sorted(Path(OUTPUT_DIR).glob("*/*"), key=lambda p: p.stat().st_mtime)
    candidates = [p for p in candidates if (p / "adapter_config.json").exists()]
    CHECKPOINT_DIR = str(candidates[-1]) if candidates else ""

if CHECKPOINT_DIR:
    print("Evaluating checkpoint:", CHECKPOINT_DIR)
    subprocess.run([
        "python", "scripts/test_infer.py",
        "--config", CONFIG_PATH,
        "--checkpoint", CHECKPOINT_DIR,
        "--split", "test_alter",
        "--output_file", "results/sail_qformer_eval_test_alter.json",
    ], check=True)
else:
    print("No checkpoint found. Train first, then rerun this cell.")


In [ ]:
import subprocess
subprocess.run(["python", "scripts/visualize_training.py", "--save", "sail_qformer_training_plot.png"], check=True)

from IPython.display import Image, display
display(Image("sail_qformer_training_plot.png"))
